In [38]:
%autosave 300
%load_ext autoreload
%autoreload 2 
%reload_ext autoreload
%config Completer.use_jedi = False

Autosaving every 300 seconds
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
import os

os.chdir(
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/copilot-model-run/code/Users/Soutrik.Chowdhury/ERAV2_Advanced_VisionTransformers"
)
print(os.getcwd())

/mnt/batch/tasks/shared/LS_root/mounts/clusters/copilot-model-run/code/Users/Soutrik.Chowdhury/ERAV2_Advanced_VisionTransformers


In [54]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import transforms

In [41]:
dummy_image = torch.randn(1, 3, 572, 572)
print(dummy_image.shape)

torch.Size([1, 3, 572, 572])


In [42]:
# double convolution block


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(ConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(
            in_channels=in_ch, out_channels=out_ch, kernel_size=3, padding=0
        )
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(
            in_channels=out_ch, out_channels=out_ch, kernel_size=3, padding=0
        )

    def forward(self, x):
        op = self.conv1(x)
        op = self.relu(op)
        op = self.conv2(op)
        op = self.relu(op)

        return op

In [43]:
conv_block = ConvBlock(3, 64)
op = conv_block(dummy_image)
print(op.shape)

torch.Size([1, 64, 568, 568])


In [70]:
# encoder block


class EncoderBlock(nn.Module):
    def __init__(self, channels=[3, 64, 128, 256, 512, 1024]):
        super(EncoderBlock, self).__init__()
        self.channels = channels
        self.encoder_blocks = nn.ModuleList(
            [
                ConvBlock(self.channels[i], self.channels[i + 1])
                for i in range(len(self.channels) - 1)
            ]
        )
        self.pool = nn.MaxPool2d(kernel_size=2)

    def forward(self, x):
        encoder_layers = []
        for i, block in enumerate(self.encoder_blocks):
            x = block(x)
            encoder_layers.append(x)
            if i < len(self.encoder_blocks):
                x = self.pool(x)

        return encoder_layers

In [75]:
encoder = EncoderBlock(channels=[3, 64, 128, 256, 512, 1024])
ftrs = encoder(dummy_image)

In [76]:
for ftr in ftrs:
    print(ftr.shape)

torch.Size([1, 64, 568, 568])
torch.Size([1, 128, 280, 280])
torch.Size([1, 256, 136, 136])
torch.Size([1, 512, 64, 64])
torch.Size([1, 1024, 28, 28])


In [77]:
class DecoderBlock(nn.Module):
    def __init__(self, channels=[1024, 512, 256, 128, 64]):
        super().__init__()
        self.channels = channels
        self.upconvol_layers = nn.ModuleList(
            [
                nn.ConvTranspose2d(
                    in_channels=self.channels[i],
                    out_channels=self.channels[i + 1],
                    kernel_size=2,
                    stride=2,
                )
                for i in range(len(self.channels) - 1)
            ]
        )
        self.conv_blocks = nn.ModuleList(
            [
                ConvBlock(channels[i], channels[i + 1])
                for i in range(len(self.channels) - 1)
            ]
        )

    def crop(self, enc_ftrs, x):
        """Crop the enc features as per target x"""
        _, _, H, W = x.shape
        enc_ftrs = transforms.CenterCrop([H, W])(enc_ftrs)
        return enc_ftrs

    def forward(self, x, encoder_features):
        for i in range(len(self.channels) - 1):
            x = self.upconvol_layers[i](x)
            enc_ftrs = self.crop(encoder_features[i], x)
            x = torch.cat([x, enc_ftrs], dim=1)
            x = self.conv_blocks[i](x)

        return x

In [78]:
decoder = DecoderBlock()
x = torch.randn(1, 1024, 28, 28)  # output from encoder block
op = decoder(x, ftrs[::-1][1:])
op.shape

torch.Size([1, 64, 388, 388])

In [81]:
# final Unet class


class UNet(nn.Module):
    def __init__(
        self,
        enc_channels=[3, 64, 128, 256, 512, 1024],
        dec_channels=[1024, 512, 256, 128, 64],
        num_class=1,
        retain_dim=False,
        out_sz=(572, 572),
    ):
        super().__init__()
        self.encoder = EncoderBlock(enc_channels)
        self.decoder = DecoderBlock(dec_channels)
        self.head = nn.Conv2d(
            in_channels=dec_channels[-1], out_channels=num_class, kernel_size=1
        )  # 1*1
        self.retain_dim = retain_dim

    def forward(self, x):
        enc_features = self.encoder(x)
        # last op of enc features goes in a input to decoder and rest all is added as skip connect in rev order
        dec_features = self.decoder(enc_features[::-1][0], enc_features[::-1][1:])
        out = self.head(dec_features)

        if self.retain_dim:
            out = F.interpolate(out, out_sz)

        return out

As mentioned before, since the convolution operations are 3x3 without padding, the output feature map size is not the same as the input feature map size. Also, as shown in fig-1 the final output is of shape 1x388x388 while the input Image had dimensions 572x572. This can create problems when calculating BCELoss in PyTorch as it expects the input and output feature maps to have the same shape.

Therefore, if we want to retain_dim, I have added F.interpolate operation to the U-Net to make the output size same as the input Image size.

In [82]:
unet = UNet()
x = torch.randn(1, 3, 572, 572)
unet(x).shape

torch.Size([1, 1, 388, 388])

References:
* https://amaarora.github.io/posts/2020-09-13-unet.html